# Evaluation Notebook

This notebook evaluates the trained EEG motor imagery models.

It loads the trained model results from the `results/` directory, compares full-channel and reduced-channel performance, shows confusion matrices, and visualizes an example model prediction.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from torch.utils.data import DataLoader

from eeg_project.dataset import (
    EEGMotorImageryDataset,
    INDEX_TO_LABEL,
)

from eeg_project.models import EEGCNN1D

In [ ]:
root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

data_dir = root / "data"
results_dir = root / "results"
figures_dir = results_dir / "figures"

figures_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", root)
print("Data directory:", data_dir)
print("Results directory:", results_dir)
print("Results directory exists:", results_dir.exists())

In [ ]:
summary_path = results_dir / "channel_comparison_seed42.csv"

if not summary_path.exists():
    raise FileNotFoundError(
        f"Missing {summary_path}. "
        "Run training first: python scripts/train_models.py --data-dir data --epochs 10"
    )

summary = pd.read_csv(summary_path)
summary

In [ ]:
plt.figure(figsize=(7, 4))

x = np.arange(len(summary))
width = 0.35

plt.bar(x - width / 2, summary["test_accuracy"], width, label="Accuracy")
plt.bar(x + width / 2, summary["test_macro_f1"], width, label="Macro F1")

plt.xticks(x, summary["channel_set"])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Full vs Reduced Channel EEG Classification")
plt.legend()
plt.tight_layout()

comparison_fig = figures_dir / "evaluation_channel_comparison.png"
plt.savefig(comparison_fig, dpi=200)
plt.show()

print("Saved:", comparison_fig)

In [ ]:
channel_set = "motor21"  # options: "full", "motor21", "central3"

result_path = results_dir / f"eeg_{channel_set}_seed42.json"

if not result_path.exists():
    raise FileNotFoundError(f"Missing result file: {result_path}")

with open(result_path, "r", encoding="utf-8") as f:
    result = json.load(f)

print("Loaded:", result_path)
print("Model path:", result["model_path"])
print("Test accuracy:", result["test_accuracy"])
print("Test macro F1:", result["test_macro_f1"])
print("Confusion matrix:", result["test_confusion_matrix"])

In [ ]:
cm = np.array(result["test_confusion_matrix"])

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[INDEX_TO_LABEL[0], INDEX_TO_LABEL[1]],
)

fig, ax = plt.subplots(figsize=(5, 4))
display.plot(ax=ax, values_format="d")

plt.title(f"Confusion Matrix: {channel_set}")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

cm_fig = figures_dir / f"evaluation_confusion_{channel_set}.png"
plt.savefig(cm_fig, dpi=200)
plt.show()

print("Saved:", cm_fig)

In [ ]:
test_subjects = result["test_subjects"]

test_dataset = EEGMotorImageryDataset(
    data_dir=data_dir,
    subjects=test_subjects,
    channel_set=channel_set,
    tmin=result["args"]["tmin"],
    tmax=result["args"]["tmax"],
    normalize=True,
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(test_dataset.summary())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EEGCNN1D(
    num_channels=result["num_channels"],
    num_classes=2,
    dropout=result["args"]["dropout"],
).to(device)

checkpoint = torch.load(result["model_path"], map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded trained model on:", device)

In [ ]:
all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)

        logits = model(batch_x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(batch_y.numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

print("Accuracy:", accuracy_score(all_targets, all_preds))
print("Macro F1:", f1_score(all_targets, all_preds, average="macro", zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(all_targets, all_preds, labels=[0, 1]))

In [ ]:
example_index = 0

x, y = test_dataset[example_index]

with torch.no_grad():
    logits = model(x.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))

metadata = test_dataset.get_metadata(example_index)

print("Metadata:", metadata)
print("True label:", y.item(), INDEX_TO_LABEL[y.item()])
print("Predicted label:", pred, INDEX_TO_LABEL[pred])
print("Probabilities:", probs)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

for i in range(min(8, x.shape[0])):
    axes[0].plot(x[i].numpy() + i * 5, label=test_dataset.channel_names[i])

axes[0].set_title(
    f"Example EEG Window | True: {INDEX_TO_LABEL[y.item()]} | Predicted: {INDEX_TO_LABEL[pred]}"
)
axes[0].set_xlabel("Time points")
axes[0].set_ylabel("Normalized amplitude + offset")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].bar([INDEX_TO_LABEL[0], INDEX_TO_LABEL[1]], probs)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Predicted probability")
axes[1].set_title("Model Prediction Probabilities")

plt.tight_layout()

prediction_fig = figures_dir / f"evaluation_example_prediction_{channel_set}.png"
plt.savefig(prediction_fig, dpi=200)
plt.show()

print("Saved:", prediction_fig)